In [5]:
import pandas as pd 
import numpy as np
import re 
import json 
import os

import kagglehub
from huggingface_hub import scan_cache_dir
import humanize

from transformers import AutoModel, AutoTokenizer
# from numind import NuMind

from gliner import GLiNER

---

# 1.0 Dataset

1. look for datasets with exercise / training notes (from notetaking)? 
2. statements should contain multiple info such as exercise type, duration, frequency etc 

2025-10-28

1. don't see any notetaking datasets on kaggle
2. potential datasets that may be useful:
    - https://www.kaggle.com/datasets/niharika41298/gym-exercise-data 
        - table of exercises and their definitions 
        - may be useful for finetuning model to recognise exercises 
    

In [ ]:
# function
def download_dataset_from_kaggle(kaggle_dir: str) -> pd.DataFrame: 
    file_dir = kagglehub.dataset_download(kaggle_dir)
    file_list = os.listdir(file_dir)

    if len(file_list) > 1: 
        raise ValueError("There are more than 1 file to download. Pick one.")
    
    file_name = file_list[0]
    file_path = os.path.join(file_dir, file_name)

    return pd.read_csv(file_path)

In [ ]:
# download csv 
df = download_dataset_from_kaggle("niharika41298/gym-exercise-data")
df.shape

In [ ]:
df.head()


In [ ]:
# save df

---

# 2.0 NLP layer / engine - NER using NuNER_Zero

In [2]:
# function 
def merge_entities(entities):
    if not entities:
        return []
    merged = []
    current = entities[0]
    for next_entity in entities[1:]:
        if next_entity['label'] == current['label'] and (next_entity['start'] == current['end'] + 1 or next_entity['start'] == current['end']):
            current['text'] = text[current['start']: next_entity['end']].strip()
            current['end'] = next_entity['end']
        else:
            merged.append(current)
            current = next_entity
    # Append the last entity
    merged.append(current)
    return merged

In [3]:
# check for cached hf models
cache_info = scan_cache_dir()

for repo in cache_info.repos:
    repo_size = sum(rev.size_on_disk for rev in repo.revisions if rev.size_on_disk)
    print(f"{repo.repo_id:<40}  {humanize.naturalsize(repo_size)}")

facebook/bart-large-mnli                  1.6 GB
distilbert/distilgpt2                     2.9 MB
numind/NuNerZero                          1.8 GB
distilbert-base-cased                     263.9 MB
dany0407/eli5_category                    115.4 MB
microsoft/deberta-v3-large                1.7 GB


In [6]:
# model 
model = GLiNER.from_pretrained("numind/NuNerZero")

/Users/ashleythoo/Documents/notetaking/.venv/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


ValueError: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434

In [8]:
import torch

In [9]:
print(torch.__version__)
print(torch.cuda.is_available())  # False
print(torch.backends.mps.is_available())  # False


2.2.2
False
True


In [10]:
!torch.__version__

zsh:1: command not found: torch.__version__
